In [35]:
import os
import sys
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

Running with 8 CPU cores | pyarrow 24.0.0


Variables

In [ ]:
%run 0_variables.ipynb

Feature ranking

In [39]:
features = pd.read_parquet(os.environ["FEATURES_PROCESSED_PATH"])

_stem = Path(os.environ['FEATURE_DATASET']).stem
future_prediction_targets_agg = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{os.environ['TARGET']}_targets_agg_{_stem}.parquet")
)

"""
Rank features

MI ranking: measures mutual information — a non-linear, information-theoretic score. It answers "does knowing this feature reduce uncertainty about the target?",
capturing non-linear dependencies too. Targets are passed in pre-aggregated to output_resolution_minutes,
so MI is computed directly at the output resolution.
"""

def rank_features(features:pd.DataFrame,
    future_prediction_targets_agg: pd.DataFrame,
    feature_selection_subsample_start: pd.Timestamp,
    feature_selection_subsample_end: pd.Timestamp,
    feature_selection_subsample_amount: int,
    output_resolution_minutes: int,
):
    
    target_rounds = 10

    feature_cols = list(features.columns)
    target_cols_agg = list(future_prediction_targets_agg.columns)
    n_buckets = len(target_cols_agg)

    def _filter_data_for_feature_time_range_subset():
        features_subset = features.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        targets_subset = future_prediction_targets_agg.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        shared_index = features_subset.index.intersection(targets_subset.index)
        return features_subset.loc[shared_index], targets_subset.loc[shared_index]

    features_subset, targets_subset = _filter_data_for_feature_time_range_subset()

    features_subset = features_subset.values.astype(np.float32)
    targets_subset = targets_subset.reset_index(drop=True)

    def _subsample_features():
        seed = np.random.default_rng(42)
        n_samples = min(feature_selection_subsample_amount, len(features_subset))
        subsample_index = seed.choice(len(features_subset), size=n_samples, replace=False)
        subsample_index.sort()
        return features_subset[subsample_index], targets_subset.iloc[subsample_index]

    X_subsample, y_subsample = _subsample_features()

    def _mutual_information_scoring():
        aggregated_target_matrix = y_subsample[target_cols_agg].values.astype(np.float32)
        n_features = X_subsample.shape[1]
        n_subsample_rows = X_subsample.shape[0]

        # Split features into chunks so n_tasks >> n_workers → multiple rounds
        n_feature_chunks = max(1, (target_rounds * _NCPU + n_buckets - 1) // n_buckets)
        chunk_edges = np.array_split(np.arange(n_features), n_feature_chunks)
        feature_chunk_ranges = [(int(c[0]), int(c[-1]) + 1) for c in chunk_edges if len(c) > 0]
        n_tasks = n_buckets * len(feature_chunk_ranges)

        print(
            f"  MI scoring: {n_features} features × {n_buckets} horizons ({output_resolution_minutes}-min) "
            f"| subsample n={n_subsample_rows:,} | {n_tasks} tasks across {_NCPU} CPUs (~{n_tasks // _NCPU} rounds)",
            flush=True,
        )

        def _score_chunk(bucket_idx, feature_start_idx, feature_end_idx):
            return bucket_idx, feature_start_idx, feature_end_idx, mutual_info_regression(
                X_subsample[:, feature_start_idx:feature_end_idx], aggregated_target_matrix[:, bucket_idx],
                discrete_features=False, n_neighbors=3, random_state=42,
            )

        gen = Parallel(n_jobs=-1, backend="loky", batch_size=1, return_as="generator_unordered")(
            delayed(_score_chunk)(bucket_idx, feature_start_idx, feature_end_idx)
            for bucket_idx in range(n_buckets) for feature_start_idx, feature_end_idx in feature_chunk_ranges
        )
        scores = np.empty((n_features, n_buckets))
        for bucket_idx, feature_start_idx, feature_end_idx, chunk_scores in tqdm(gen, total=n_tasks, desc="MI scoring", leave=True):
            scores[feature_start_idx:feature_end_idx, bucket_idx] = chunk_scores

        mi_matrix = pd.DataFrame(scores, index=feature_cols, columns=target_cols_agg)
        feature_cols_ranked = mi_matrix.mean(axis=1).sort_values(ascending=False)
        return feature_cols_ranked, mi_matrix

    feature_cols_ranked, mi_matrix = _mutual_information_scoring()

    df = pd.DataFrame({
        "rank": range(1, len(feature_cols_ranked) + 1),
        "feature": feature_cols_ranked.index,
        "mean_mi": feature_cols_ranked.values,
        "target": os.environ["TARGET"],
        "feature_dataset": Path(os.environ["FEATURE_DATASET"]).stem,
    }).set_index("feature")

    df_horizons = mi_matrix.reindex(feature_cols_ranked.index)
    feature_data = pd.concat([df, df_horizons], axis=1).reset_index(names="feature")

    display(feature_data[:3])
    return feature_data

feature_data = rank_features(
    features=features,
    future_prediction_targets_agg=future_prediction_targets_agg,
    feature_selection_subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_START"]),
    feature_selection_subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_END"]),
    feature_selection_subsample_amount=int(os.environ["FEATURE_RANKING_SUBSAMPLE_AMOUNT"]),
    output_resolution_minutes=int(os.environ["OUTPUT_RESOLUTION"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
feature_data.to_parquet(
    resolve(f"4_Features select/Selected_features/{os.environ['TARGET']}_feature_data_{_stem}.parquet")
)
